# Designer Resume Paraphrasing

This notebook randomly samples 10 resumes from `Resume_DESIGNER.csv`, asks the OpenAI API for 4 faithful paraphrases of each resume, and saves the original text plus all 4 variants to JSON.

Requirements:
- `OPENAI_API_KEY` must be available in the environment.
- The CSV at the configured path must be the real dataset, not a Git LFS pointer file.


In [ ]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

PROJECT_ROOT = Path("/Users/qihanwang/Desktop/GenAI_Accountability/project")
DATASET_PATH = PROJECT_ROOT / "resume_project/datasets/Tech_Designer/Resume_DESIGNER.csv"
OUTPUT_PATH = PROJECT_ROOT / "resume_project/datasets/Tech_Designer/designer_resume_paraphrases_sample_10.json"

MODEL = "gpt-5.4-mini"
SAMPLE_SIZE = 10
PARAPHRASES_PER_RESUME = 4
RANDOM_SEED = 42
MAX_RETRIES = 3
RESUME_TEXT_COLUMN = "Resume_str"

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("OPENAI_API_KEY is not set in the environment.")

client = OpenAI(api_key=api_key)

print(f"Dataset path: {DATASET_PATH}")
print(f"Output path:  {OUTPUT_PATH}")
print(f"Model:        {MODEL}")


In [ ]:
def assert_not_lfs_pointer(path: Path) -> None:
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        first_line = handle.readline().strip()

    if first_line == "version https://git-lfs.github.com/spec/v1":
        raise RuntimeError(
            "The dataset file is currently a Git LFS pointer, not the real CSV. "
            "Replace it with the actual dataset contents before running this notebook."
        )


def load_sample(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    assert_not_lfs_pointer(path)
    df = pd.read_csv(path)

    required_columns = {"ID", RESUME_TEXT_COLUMN, "Category"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

    cleaned = df[["ID", "Category", RESUME_TEXT_COLUMN]].copy()
    cleaned[RESUME_TEXT_COLUMN] = cleaned[RESUME_TEXT_COLUMN].fillna("").astype(str)
    cleaned = cleaned[cleaned[RESUME_TEXT_COLUMN].str.strip() != ""].reset_index(drop=True)

    if len(cleaned) < SAMPLE_SIZE:
        raise ValueError(
            f"Need at least {SAMPLE_SIZE} non-empty resumes, but found {len(cleaned)}."
        )

    sample = cleaned.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
    return sample


sample_df = load_sample(DATASET_PATH)
sample_df.head(2)


In [ ]:
PARAPHRASE_SCHEMA = {
    "type": "object",
    "properties": {
        "paraphrase_1": {"type": "string"},
        "paraphrase_2": {"type": "string"},
        "paraphrase_3": {"type": "string"},
        "paraphrase_4": {"type": "string"},
    },
    "required": ["paraphrase_1", "paraphrase_2", "paraphrase_3", "paraphrase_4"],
    "additionalProperties": False,
}

SYSTEM_PROMPT = """You are a meticulous resume paraphrasing assistant.
Your job is to rewrite a resume into four distinct paraphrases.

Hard requirements:
- Preserve every piece of meaning from the original text.
- Do not add, remove, summarize, infer, normalize, or embellish anything.
- Keep all facts, numbers, dates, contact details, technologies, tools, employers, schools, degrees, locations, and achievements.
- Only change grammar, wording, phrasing, and local sentence structure.
- Keep the output in English.
- Return JSON only.
"""


def paraphrase_resume(resume_text: str) -> dict:
    user_prompt = f"""Generate exactly four faithful paraphrases of the resume below.

The paraphrases must differ in wording and grammar, but they must preserve all meaning and all details.

Resume:
<<RESUME_START>>
{resume_text}
<<RESUME_END>>
"""

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.responses.create(
                model=MODEL,
                input=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "resume_paraphrases",
                        "strict": True,
                        "schema": PARAPHRASE_SCHEMA,
                    }
                },
            )
            return json.loads(response.output_text)
        except Exception:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(attempt * 2)

    raise RuntimeError("Unexpected retry flow.")


In [ ]:
results = []

for index, row in sample_df.iterrows():
    paraphrases = paraphrase_resume(row[RESUME_TEXT_COLUMN])
    results.append(
        {
            "resume_id": int(row["ID"]),
            "category": row["Category"],
            "original": row[RESUME_TEXT_COLUMN],
            "paraphrase_1": paraphrases["paraphrase_1"],
            "paraphrase_2": paraphrases["paraphrase_2"],
            "paraphrase_3": paraphrases["paraphrase_3"],
            "paraphrase_4": paraphrases["paraphrase_4"],
        }
    )
    print(f"Completed {index + 1}/{len(sample_df)} resumes")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Saved {len(results)} records to {OUTPUT_PATH}")
results[0]
